## Solving RL problems with Evolutionary Computing

### Getting started with Gymnasium Gym

Simply install gymnasium using pip:

pip install gymnasium stable-baselines3


In [1]:
import gymnasium as gym
import time

### Mountain Car Gym

We are going to use the Mountain Car gym. See: https://gym.openai.com/envs/MountainCar-v0/

In [2]:
env = gym.make('MountainCar-v0', render_mode="human") 
env.reset()
for _ in range(500):#number of steps per episode
    env.render() #display on external window
    action=env.action_space.sample() #make random action
    obs, rew, done, _,info=env.step(action) # take a random action
    time.sleep(0.02)
    if done:
       env.reset()
env.close()

### Observations
If we ever want to do better than take random actions at each step, it’d probably be good to actually know what our actions are doing to the environment.

The environment’s `step` function returns exactly what we need. In fact, step returns four values. These are:

- **observation** (object): an environment-specific object representing your observation of the environment. For example, pixel data from a camera, joint angles and joint velocities of a robot, or the board state in a board game.

- **reward** (float): amount of reward achieved by the previous action. The scale varies between environments, but the goal is always to increase your total reward.

- **done** (boolean): whether it’s time to reset the environment again. Most (but not all) tasks are divided up into well-defined episodes, and done being True indicates the episode has terminated. (For example, perhaps the pole tipped too far, or you lost your last life.)

- **info** (dict): diagnostic information useful for debugging. It can sometimes be useful for learning (for example, it might contain the raw probabilities behind the environment’s last state change).  However, official evaluations of your agent are not allowed to use this for learning.

This is just an implementation of the classic “agent-environment loop”. Each timestep, the agent chooses an action, and the environment returns an observation and a reward.

See: https://github.com/openai/gym/wiki/MountainCar-v0

### Evolutionary approach

We want to give a set of actions (as evolutionary individual) to the gym environment and evaluate its effectiveness (as fitness). The actions available for MountainCar environment is 0,1 or 2 (left,none, right). 
Actions:
- 0      Accelerate to the Left
- 1      Don't accelerate
- 2      Accelerate to the Right

Let's imagine this set of actions: 0,0,0,0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0,0,0,0,0, 2, 2, 2, 
         2, 2, 2, 2, 2, 2, 2, 2, 2,2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2 . 
Here is how we can do it:

In [3]:
#this would be an Individual example
actions=[0,0,0,0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0,0,0,0,0, 2, 2, 2, 
         2, 2, 2, 2, 2, 2, 2, 2, 2,2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2] 
env = gym.make('MountainCar-v0', render_mode="human")
env.reset()
# feed the actions to the environment:
for action in actions:
    env.render()
    # provide an action and get feedback:
    observation, reward, done, _,info = env.step(action)
    time.sleep(0.1)
    # episode over 
    if done:
        env.reset()
env.close()

### Fitness

How would we calculate the fitness? The reward value from the environment would be able to give sufficient information because its value will always be -200 if we don't hit the flag.

When we compare two candidate solutions that don't hit the flag, we would still like to know which one got closer to it and consider it a better solution. Therefore, we are going to use the final position of the car, in addition to the reward value, to determine the score of the solution. 

If the car did not hit the flag, the score will be the distance from the flag. 

If the car hits the flag, the base score will be zero, and from that, we deduct an additional value based on how many steps were left, making the score negative. 

We are going to calculate the fitness score at the end of each episode given by an Individual. 

The observation array stores:


        Index    Observation               Min            Max
        0        Car Position              -1.2           0.6
        1        Car Velocity              -0.07          0.07

In [4]:
MAX_STEPS = 48 env.reset()
FLAG_LOCATION = 0.5

actions=[0,0,0,0,0,0,0,0,0,0,0,0, 0,0,0,0,0,0,0,0,0,0,0,0, 2, 2, 2, 
         2, 2, 2, 2, 2, 2, 2, 2, 2,2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2] # 48 actions steps

env = gym.make('MountainCar-v0', render_mode="human")
env.reset()

actionCounter = 0
for action in actions:
    actionCounter+=1
    env.render()
    # provide an action and get feedback:
    observation, reward, done, _,info = env.step(action)
    print(observation)
    time.sleep(0.1)
    # episode over 
    if done:
        break
        
env.close()

# evaluate the results to produce the score:
if actionCounter < MAX_STEPS:

    # the car hit the flag:
    # start from a score of 0
    # reward further for a smaller amount of steps
    score = 0 - (MAX_STEPS - actionCounter)/MAX_STEPS
else:
    # the car did not hit the flag:
    # reward according to distance from flag
    score = abs(observation[0] - FLAG_LOCATION)  # we want to minimize that

print((MAX_STEPS - actionCounter)/MAX_STEPS)
print(observation[0]) #last observation car position
print(score)

[-0.53513977 -0.00092036]
[-0.53697359 -0.00183382]
[-0.53970712 -0.00273353]
[-0.54331989 -0.00361277]
[-0.54778484 -0.00446495]
[-0.55306855 -0.00528371]
[-0.55913152 -0.00606297]
[-0.56592851 -0.00679698]
[-0.57340887 -0.00748036]
[-0.58151705 -0.00810818]
[-0.59019302 -0.00867597]
[-0.59937285 -0.00917983]
[-0.60898926 -0.00961641]
[-0.61897222 -0.00998296]
[-0.6292496  -0.01027738]
[-0.63974779 -0.0104982 ]
[-0.6503924 -0.0106446]
[-0.66110881 -0.01071642]
[-0.67182291 -0.0107141 ]
[-0.6824616  -0.01063869]
[-0.69295341 -0.01049181]
[-0.70322901 -0.0102756 ]
[-0.71322165 -0.00999264]
[-0.72286758 -0.00964594]
[-0.73010645 -0.00723886]
[-0.73489369 -0.00478724]
[-0.73720024 -0.00230655]
[-7.37012184e-01  1.88054441e-04]
[-0.73433066  0.00268152]
[-0.72917184  0.00515882]
[-0.72156712  0.00760472]
[-0.71156339  0.01000373]
[-0.69922346  0.01233993]
[-0.68462645  0.01459701]
[-0.66786815  0.01675829]
[-0.6490613   0.01880685]
[-0.62833555  0.02072576]
[-0.60583713  0.02249842]
[-0.58

### GetScore as Fitness

We wrap the above code with def function and remove the rendering function.

In [ ]:
MAX_STEPS = 200
FLAG_LOCATION = 0.5
env = gym.make('MountainCar-v0', render_mode="human")
env.reset()

In [16]:
def getScore(render,actions):
    env.reset()
    actionCounter = 0
    for action in actions:
        actionCounter+=1
        if(render):
            env.render() #remove display
            time.sleep(0.1)
        # provide an action and get feedback:
        observation, reward, done, _,info = env.step(action)

        # episode over 
        if done:
            break
            
    
    env.close()

    # evaluate the results to produce the score:
    if actionCounter < MAX_STEPS:
        # the car hit the flag:
        # start from a score of 0
        # reward further for a smaller amount of steps
        score = 0 - (MAX_STEPS - actionCounter)/MAX_STEPS
    else:
        # the car did not hit the flag:
        # reward according to distance from flag
        score = abs(observation[0] - FLAG_LOCATION)  # we want to minimize that
    return score

In [ ]:
actions=[0,0,0,0,0,0,0,0,0,0,2,2,2,2,2,0,0,0,0, 0,0,0,0,2,2,2,2,2,0,0,0,0,0,0,0,0, 2, 2, 2, 
         2, 2, 2, 2, 2, 2, 2, 2, 2,2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2] 
print(getScore(True,actions))

### Get Cracking with DEAP

In [29]:
from deap import base
from deap import creator
from deap import tools
from deap import algorithms

import random
import numpy

# Genetic Algorithm constants:
POPULATION_SIZE = 50
P_CROSSOVER = 0.9  # probability for crossover
P_MUTATION = 0.5   # probability for mutating an individual
MAX_GENERATIONS = 5
HALL_OF_FAME_SIZE = 1
# set the random seed:
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

toolbox = base.Toolbox()

In [30]:
# define a single objective, minimizing fitness strategy:
creator.create("FitnessMin", base.Fitness, weights=(-1.0,))

# create the Individual class based on list:
creator.create("Individual", list, fitness=creator.FitnessMin)

# create an operator that randomly returns 0, 1 or 2:
#YOURCODE

# create an operator that generates an individual:individualCreator
#YOURCODE


# create the population operator to generate a list of individuals:
toolbox.register("populationCreator", tools.initRepeat, list, toolbox.individualCreator)

# fitness calculation requires return a tuple
def getCarScore(individual):
    #YOURCODE


toolbox.register("evaluate", getCarScore)

# genetic operators for binary list:
#YOURCODE select, mate, mutate

# create initial population (generation 0):
population = toolbox.populationCreator(n=POPULATION_SIZE)

In [ ]:
#reset the enviroment to no display rendering for the GA to process
env = gym.make('MountainCar-v0')
env.reset()

In [31]:
# prepare the statistics object:
stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("min", numpy.min)
stats.register("avg", numpy.mean)

# define the hall-of-fame object:
hof = tools.HallOfFame(HALL_OF_FAME_SIZE)
population, logbook = algorithms.eaSimple(population,
                                                      toolbox,
                                                      cxpb=P_CROSSOVER,
                                                      mutpb=P_MUTATION,
                                                      ngen=MAX_GENERATIONS,
                                                      stats=stats,
                                                      halloffame=hof,
                                                      verbose=True)

# print best solution:
best = hof.items[0]
print()
print("Best Solution = ", best)
print("Best Fitness = ", best.fitness.values[0])

gen	nevals	min     	avg    
0  	50    	0.709752	1.02241
1  	48    	0.740078	0.972556
2  	50    	0.73236 	0.944281
3  	44    	0.761174	0.892763
4  	46    	0.587467	0.872738
5  	48    	0.531391	0.822935
6  	48    	0.622386	0.786406
7  	42    	0.533966	0.749076
8  	46    	0.542762	0.712523
9  	47    	0.525546	0.678524
10 	47    	0.540387	0.649526
11 	47    	0.529597	0.628324
12 	49    	0.496323	0.611992
13 	45    	0.482582	0.575592
14 	47    	0.333719	0.561113
15 	48    	0.388419	0.535321
16 	47    	0.329224	0.509364
17 	47    	0.299307	0.495677
18 	49    	0.303856	0.457271
19 	50    	0.296923	0.444452
20 	48    	0.297222	0.434013
21 	48    	0.297222	0.423527
22 	47    	0.291386	0.419649
23 	47    	0.297826	0.395235
24 	49    	0.301843	0.387231
25 	50    	0.288222	0.379244
26 	50    	0.280011	0.356913
27 	48    	0.285024	0.365358
28 	46    	0.269267	0.365237
29 	49    	0.281201	0.363921
30 	45    	0.233239	0.354451
31 	50    	0.266153	0.35718 
32 	47    	0.262629	0.346121
33 	49    	0.268

### Replay action

We can replay the action by feeding the hof item back into the gym environment.

In [ ]:
#reset back the environment to display rendering
env = gym.make('MountainCar-v0', render_mode="human")
env.reset()

In [32]:
getScore(True, best)

0.23308492566002925